# Microbots: a GNN + GRU policy, trained in Colab

Every bot runs the same small network. Each step it:

1. **sees** only its own 5×5×5 block (which cells hold a bot, which belong to the shape) and whether it's locked in place,
2. **talks** to the bots touching it, a few rounds per step (a **graph neural network**, GNN),
3. **remembers** with a hidden state it carries from step to step (a **GRU**),
4. **picks** one of 19 moves: stay, 6 slides, 12 flips. Illegal moves are masked out.

The first trained policy in this repo was handed two hand-made inputs: a "how far to the work" signal relayed between bots, and a compass. **This one gets neither.** The question: can the bots learn to pass that information to each other themselves?

Training is imitation: copy the hand-written rules (which use a map of the whole grid no bot can see), then DAgger (the network drives, the rules label). Code: [`linking/learn/gnn_gru.py`](https://github.com/ItsAdarsh008/microbots/blob/main/linking/learn/gnn_gru.py). Textbook: Linking 2.

**Before you start:** Runtime → Change runtime type → **T4 GPU**. Recording and evaluation run the simulation on the CPU either way; the GPU speeds up training.

In [ ]:
!nvidia-smi -L || echo "No GPU: Runtime > Change runtime type > T4 GPU (it still works on CPU, just slower)"
!git clone -q --depth 1 https://github.com/ItsAdarsh008/microbots
%cd microbots/linking/learn
!pip -q install scipy
import jax, sys
print("JAX", jax.__version__, "on", jax.devices())

## 1. Record the teacher

The hand-written rules build the **wall** (`sheet`) and the **arch** a dozen times each. At every step we save, for every bot: what it sees (251 numbers), who it can hear (the bots in the 26 cells around it), its legal moves, and the move the rules would make. About 5 seconds per build.

In [ ]:
import time, numpy as np, jax, jax.numpy as jnp
import gnn_gru as G

SEEDS = 12                     # builds per shape; fewer = faster, more = better
builds = []
t0 = time.time()
for shape in ["sheet", "arch"]:
    for seed in range(SEEDS):
        rec, ok, steps = G.rollout(shape, seed)
        builds.append(rec)
        print(f"{shape:5s} seed {seed:2d}: {steps} steps")
print(f"{len(builds)} builds in {time.time() - t0:.0f}s")

## 2. The network

- **Encoder:** 251 inputs → 64 numbers.
- **3 rounds of message passing:** each bot sends a message (built from its own state, the listener's state, and which direction it's in) to every bot touching it; each bot averages what it hears and updates. After 3 rounds, information has travelled 3 cells.
- **GRU memory:** the result updates the bot's hidden state, which it keeps for the next step. Over many steps, information can travel much further than 3 cells.
- **Head:** 19 move scores.

`MEMORY = False` wipes the hidden state every step, as a comparison: same network, no memory.

In [ ]:
MEMORY = True
params = G.init(jax.random.PRNGKey(0))
opt = G.adam_init(params)
log = {"loss": [], "epoch_end": [], "agree": [], "stage": []}
print(f"{G.count(params):,} weights")

## 3. Train by imitation

Each step of training runs 4 builds through 32 simulation steps at once, carrying each bot's memory through them (truncated backpropagation through time).

In [ ]:
EPOCHS = 60
t0 = time.time()
params, opt = G.train(params, opt, builds, EPOCHS, MEMORY, log)
print(f"{time.time() - t0:.0f}s")

## 4. DAgger: let the network drive

Copying alone breaks down once the network makes a mistake and ends up somewhere the rules never went. So the network drives, the rules say what they would have done at every step, and we train on that too.

In [ ]:
for r in range(2):
    new = []
    for shape in ["sheet", "arch"]:
        for seed in range(1000 + 10 * r, 1000 + 10 * r + SEEDS // 2):
            rec, ok, steps = G.rollout(shape, seed, G.Driver(params, MEMORY, seed), max_steps=150)
            new.append(rec)
            print(f"round {r + 1} student {shape:5s} seed {seed}: {steps} steps, finished={ok}")
    builds += new
    params, opt = G.train(params, opt, builds, 15, MEMORY, log, stage=f"dagger {r + 1}")

## 5. The training curve

In [ ]:
import matplotlib.pyplot as plt
loss = np.array(log["loss"])
fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(loss, color="#a9c1df", lw=0.6)
ax.plot(np.convolve(loss, np.ones(20) / 20, mode="valid"), color="#2a5ea8", lw=2)
ax.set_xlabel("training step"); ax.set_ylabel("loss")
ax2 = ax.twinx(); ax2.plot(np.array(log["epoch_end"]), np.array(log["agree"]) * 100, "o-", color="#d9622b")
ax2.set_ylabel("% same move as the rules", color="#d9622b")
plt.title("GNN + GRU policy"); plt.show()

## 6. Let it build

Unseen seeds; the **tower** was never in training. Each line: how many of 5 builds finished, and in how many steps (`None` = didn't finish in 300). Nothing ever falls: the safety check (the holder tree) still runs underneath, whatever the network picks.

For comparison, on these seeds the hand-written rules finish everything (wall ~50–65 steps, arch ~100–150, tower ~60–70), and the first MLP policy, *with* the relayed signal and compass, does too.

In [ ]:
results = G.evaluate(params, MEMORY)

## 7. Try it without memory

Run sections 2–6 again with `MEMORY = False`. If the bots really are using their memory to pass "where's the work" along, this version should do noticeably worse, especially on the arch and the tower, where the work is further from most bots.

## 8. Keep the weights

In [ ]:
import pickle
name = "gnn_gru.pkl" if MEMORY else "gnn_nomem.pkl"
pickle.dump(jax.tree_util.tree_map(np.asarray, params), open(name, "wb"))
try:
    from google.colab import files; files.download(name)
except ImportError:
    print("saved", name)